<a href="https://colab.research.google.com/github/mloporchio/Crypto-Lab-IRPET/blob/main/hash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cryptographic hash functions

This notebook will provide some examples of hash function computation.

Let's start by installing the Pycryptodome (https://www.pycryptodome.org) Python library.

In [ ]:
%%capture
!pip install pycryptodome

Then we need to import the required modules. Some of them are already part of the standard set of Python modules.

In [ ]:
from Crypto.Hash import SHA256, SHA512, SHA3_256, SHA3_512, HMAC
import binascii
import os
import requests
import time
import string

## Hashing a simple message

First, we define a utility function to hash strings of text using the SHA-256 algorithm.

Then we compute the digest of a simple "hello world" string.

In [ ]:
def sha256(data: str) -> str:
    h = SHA256.new()
    h.update(data.encode('utf-8'))
    return h.hexdigest()

msg = "hello world"
hash = sha256(msg)
print("{} -> {}".format(msg, hash))

hello world -> b94d27b9934d3e08a52e52d7da7dabfac484efe37a5380ee9088f7ace2efcde9


SHA-512 produces a longer output (i.e., 512 bits instead of 256 bits).

In [ ]:
def sha512(data: str) -> str:
    h = SHA512.new()
    h.update(data.encode('utf-8'))
    return h.hexdigest()

msg = "hello world"
hash = sha512(msg)
print("{} -> {}".format(msg, hash))

hello world -> 309ecc489c12d6eb4cc40f50c902f2b4d0ed77ee511a7c7a9bcd3ca86d4cd86f989dd35bc5ff499670da34255b45b0cfd830e81f605dcf7dc5542e93ae9cd76f


We can also measure the time it takes to produce these digests.

SHA-512 will typically take a bit less time than its counterpart. Why?

See: https://eprint.iacr.org/2010/548.pdf

The reason why SHA-512 is faster than SHA-256 on 64-bit machines is that has 37.5% less rounds per byte (80 rounds operating on 128 byte blocks) compared to SHA-256 (64 rounds operating on 64 byte blocks), where the
operations use 64-bit integer arithmetic.

In [ ]:
start1 = time.time_ns()
hash1 = sha256(msg)
end1 = time.time_ns()

start2 = time.time_ns()
hash2 = sha512(msg)
end2 = time.time_ns()

print("SHA-256:", hash1)
print("SHA-512:", hash2)

print("SHA-256 time:", end1 - start1, "ns")
print("SHA-512 time:", end2 - start2, "ns")

SHA-256: b94d27b9934d3e08a52e52d7da7dabfac484efe37a5380ee9088f7ace2efcde9
SHA-512: 309ecc489c12d6eb4cc40f50c902f2b4d0ed77ee511a7c7a9bcd3ca86d4cd86f989dd35bc5ff499670da34255b45b0cfd830e81f605dcf7dc5542e93ae9cd76f
SHA-256 time: 1731335 ns
SHA-512 time: 248802 ns


## Hashing a file

What if I want to compute the hash of a file? Remember that cryptographic hash function have an arbitrary input size.

First, let's download the file. We will download the entire "Othello" by William Shakespeare.

In [ ]:
file_url = "https://raw.githubusercontent.com/cobanov/shakespeare-dataset/refs/heads/main/text/othello_TXT_FolgerShakespeare.txt"
file_name = "othello.txt"

download = requests.get(file_url)
with open(file_name, "wb") as file:
    file.write(download.content)

Now, we can open the file and read each byte into a variable. Then we pass that variable to the SHA-256 hashing engine.

In [ ]:
with open(file_name, "rb") as file:
    data = file.read()

print("File size:", len(data), "bytes") # print the file size

h = SHA256.new()
h.update(data)
print("File hash:", h.hexdigest())

File size: 159411 bytes
File hash: 6b654ec71126492b0061fd8c995f6a5f511360841fadac07e7cd6810f766d722


Notice that I did not use my custom sha256() function. Why?

## Avalanche effect

According to this effect, a small change in the input causes a big change in the output.

In [ ]:
msg1 = "hello world"
msg2 = "hello world!"

hash1 = sha256(msg1)
hash2 = sha256(msg2)

print("SHA-256 msg1:", hash1)
print("SHA-256 msg2:", hash2)

SHA-256 msg1: b94d27b9934d3e08a52e52d7da7dabfac484efe37a5380ee9088f7ace2efcde9
SHA-256 msg2: 7509e5bda0c762d2bac7f90d758b5b2263fa01ccbc542ab5e3df163be08e6ca9


## HMAC

Let's see how it is possible to compute a MAC using the HMAC standard and the SHA-256 hash function.

In [ ]:
key = "othello"
msg = "see you at 8 pm"

key_bytes = key.encode('utf-8')
msg_bytes = msg.encode('utf-8')

h = HMAC.new(key_bytes, msg=msg_bytes, digestmod=SHA256)
tag = h.hexdigest()

print("HMAC:", tag)

HMAC: 5162606b2b5489a8d3aee67ac218ce9bbd80e555e747b7ed3403c09e679bb0b0


In [ ]:
msg_received = msg.encode('utf-8')
key_recipient = key.encode('utf-8')

h = HMAC.new(key_recipient, digestmod=SHA256)
h.update(msg_received)

try:
  h.hexverify(tag)
  print("The message '%s' is authentic" % msg)
except ValueError:
  print("The message or the key is wrong")

The message 'see you at 8 pm' is authentic


Let's see an example of brute force attack.

In [ ]:
import itertools

# ---- This is what the sender knows ----
key = "test"
msg = "see you at 8 pm"
key_bytes = key.encode('utf-8')
msg_bytes = msg.encode('utf-8')
h = HMAC.new(key_bytes, msg=msg_bytes, digestmod=SHA256)
target_tag = h.hexdigest()
print("Target HMAC:", target_tag)

# ---- Brute force (assumes that strings have length 4 and are all lowercase) ----
alphabet = string.ascii_lowercase

for candidate in itertools.product(alphabet, repeat=4):
    guess_key = ''.join(candidate).encode('utf-8')

    h2 = HMAC.new(guess_key, msg=msg_bytes, digestmod=SHA256)
    guess_tag = h2.hexdigest()

    if guess_tag == target_tag:
        print("Key found:", guess_key.decode())
        break

Target HMAC: 9440ef98d1a1d36e1b7e8ad93cbddf48f1b15cca3de5ef01218a7a3442a9dfab
Key found: test
